In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchinfo import summary

IN_CHANNELS = 3
N_CLASSES = 6


class SegNet(nn.Module):
    # SegNet network
    @staticmethod
    def weight_init(m):
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")

    def __init__(self, in_channels=IN_CHANNELS, out_channels=N_CLASSES):
        super(SegNet, self).__init__()
        self.pool = nn.MaxPool2d(2, return_indices=True)
        self.unpool = nn.MaxUnpool2d(2)

        self.conv1_1 = nn.Conv2d(in_channels, 64, 3, padding=1)
        self.conv1_1_bn = nn.BatchNorm2d(64)
        self.conv1_2 = nn.Conv2d(64, 64, 3, padding=1)
        self.conv1_2_bn = nn.BatchNorm2d(64)

        self.conv2_1 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv2_1_bn = nn.BatchNorm2d(128)
        self.conv2_2 = nn.Conv2d(128, 128, 3, padding=1)
        self.conv2_2_bn = nn.BatchNorm2d(128)

        self.conv3_1 = nn.Conv2d(128, 256, 3, padding=1)
        self.conv3_1_bn = nn.BatchNorm2d(256)
        self.conv3_2 = nn.Conv2d(256, 256, 3, padding=1)
        self.conv3_2_bn = nn.BatchNorm2d(256)
        self.conv3_3 = nn.Conv2d(256, 256, 3, padding=1)
        self.conv3_3_bn = nn.BatchNorm2d(256)

        self.conv4_1 = nn.Conv2d(256, 512, 3, padding=1)
        self.conv4_1_bn = nn.BatchNorm2d(512)
        self.conv4_2 = nn.Conv2d(512, 512, 3, padding=1)
        self.conv4_2_bn = nn.BatchNorm2d(512)
        self.conv4_3 = nn.Conv2d(512, 512, 3, padding=1)
        self.conv4_3_bn = nn.BatchNorm2d(512)

        self.conv5_1 = nn.Conv2d(512, 512, 3, padding=1)
        self.conv5_1_bn = nn.BatchNorm2d(512)
        self.conv5_2 = nn.Conv2d(512, 512, 3, padding=1)
        self.conv5_2_bn = nn.BatchNorm2d(512)
        self.conv5_3 = nn.Conv2d(512, 512, 3, padding=1)
        self.conv5_3_bn = nn.BatchNorm2d(512)

        self.conv5_3_D = nn.Conv2d(512, 512, 3, padding=1)
        self.conv5_3_D_bn = nn.BatchNorm2d(512)
        self.conv5_2_D = nn.Conv2d(512, 512, 3, padding=1)
        self.conv5_2_D_bn = nn.BatchNorm2d(512)
        self.conv5_1_D = nn.Conv2d(512, 512, 3, padding=1)
        self.conv5_1_D_bn = nn.BatchNorm2d(512)

        self.conv4_3_D = nn.Conv2d(512, 512, 3, padding=1)
        self.conv4_3_D_bn = nn.BatchNorm2d(512)
        self.conv4_2_D = nn.Conv2d(512, 512, 3, padding=1)
        self.conv4_2_D_bn = nn.BatchNorm2d(512)
        self.conv4_1_D = nn.Conv2d(512, 256, 3, padding=1)
        self.conv4_1_D_bn = nn.BatchNorm2d(256)

        self.conv3_3_D = nn.Conv2d(256, 256, 3, padding=1)
        self.conv3_3_D_bn = nn.BatchNorm2d(256)
        self.conv3_2_D = nn.Conv2d(256, 256, 3, padding=1)
        self.conv3_2_D_bn = nn.BatchNorm2d(256)
        self.conv3_1_D = nn.Conv2d(256, 128, 3, padding=1)
        self.conv3_1_D_bn = nn.BatchNorm2d(128)

        self.conv2_2_D = nn.Conv2d(128, 128, 3, padding=1)
        self.conv2_2_D_bn = nn.BatchNorm2d(128)
        self.conv2_1_D = nn.Conv2d(128, 64, 3, padding=1)
        self.conv2_1_D_bn = nn.BatchNorm2d(64)

        self.conv1_2_D = nn.Conv2d(64, 64, 3, padding=1)
        self.conv1_2_D_bn = nn.BatchNorm2d(64)
        self.conv1_1_D = nn.Conv2d(64, out_channels, 3, padding=1)

        self.apply(self.weight_init)

    def forward(self, x):
        # Encoder block 1
        x = self.conv1_1_bn(F.relu(self.conv1_1(x)))
        x = self.conv1_2_bn(F.relu(self.conv1_2(x)))
        x, mask1 = self.pool(x)

        # Encoder block 2
        x = self.conv2_1_bn(F.relu(self.conv2_1(x)))
        x = self.conv2_2_bn(F.relu(self.conv2_2(x)))
        x, mask2 = self.pool(x)

        # Encoder block 3
        x = self.conv3_1_bn(F.relu(self.conv3_1(x)))
        x = self.conv3_2_bn(F.relu(self.conv3_2(x)))
        x = self.conv3_3_bn(F.relu(self.conv3_3(x)))
        x, mask3 = self.pool(x)

        # Encoder block 4
        x = self.conv4_1_bn(F.relu(self.conv4_1(x)))
        x = self.conv4_2_bn(F.relu(self.conv4_2(x)))
        x = self.conv4_3_bn(F.relu(self.conv4_3(x)))
        x, mask4 = self.pool(x)

        # Encoder block 5
        x = self.conv5_1_bn(F.relu(self.conv5_1(x)))
        x = self.conv5_2_bn(F.relu(self.conv5_2(x)))
        x = self.conv5_3_bn(F.relu(self.conv5_3(x)))
        x, mask5 = self.pool(x)

        # Decoder block 5
        x = self.unpool(x, mask5)
        x = self.conv5_3_D_bn(F.relu(self.conv5_3_D(x)))
        x = self.conv5_2_D_bn(F.relu(self.conv5_2_D(x)))
        x = self.conv5_1_D_bn(F.relu(self.conv5_1_D(x)))

        # Decoder block 4
        x = self.unpool(x, mask4)
        x = self.conv4_3_D_bn(F.relu(self.conv4_3_D(x)))
        x = self.conv4_2_D_bn(F.relu(self.conv4_2_D(x)))
        x = self.conv4_1_D_bn(F.relu(self.conv4_1_D(x)))

        # Decoder block 3
        x = self.unpool(x, mask3)
        x = self.conv3_3_D_bn(F.relu(self.conv3_3_D(x)))
        x = self.conv3_2_D_bn(F.relu(self.conv3_2_D(x)))
        x = self.conv3_1_D_bn(F.relu(self.conv3_1_D(x)))

        # Decoder block 2
        x = self.unpool(x, mask2)
        x = self.conv2_2_D_bn(F.relu(self.conv2_2_D(x)))
        x = self.conv2_1_D_bn(F.relu(self.conv2_1_D(x)))

        # Decoder block 1
        x = self.unpool(x, mask1)
        x = self.conv1_2_D_bn(F.relu(self.conv1_2_D(x)))
        # x = F.log_softmax(self.conv1_1_D(x), dim=1)
        x = self.conv1_1_D(x)
        return x


model = SegNet(in_channels=IN_CHANNELS, out_channels=N_CLASSES)
summary(model, input_size=(1, IN_CHANNELS, 256, 256))

Layer (type:depth-idx)                   Output Shape              Param #
SegNet                                   [1, 6, 256, 256]          --
├─Conv2d: 1-1                            [1, 64, 256, 256]         1,792
├─BatchNorm2d: 1-2                       [1, 64, 256, 256]         128
├─Conv2d: 1-3                            [1, 64, 256, 256]         36,928
├─BatchNorm2d: 1-4                       [1, 64, 256, 256]         128
├─MaxPool2d: 1-5                         [1, 64, 128, 128]         --
├─Conv2d: 1-6                            [1, 128, 128, 128]        73,856
├─BatchNorm2d: 1-7                       [1, 128, 128, 128]        256
├─Conv2d: 1-8                            [1, 128, 128, 128]        147,584
├─BatchNorm2d: 1-9                       [1, 128, 128, 128]        256
├─MaxPool2d: 1-10                        [1, 128, 64, 64]          --
├─Conv2d: 1-11                           [1, 256, 64, 64]          295,168
├─BatchNorm2d: 1-12                      [1, 256, 64, 64]   

In [2]:
import lightning as L
from torchmetrics.classification import MulticlassF1Score
from torchmetrics.segmentation import MeanIoU
import torchio as tio
from torch.utils.data import DataLoader

L.seed_everything(42)


class Segmentator(L.LightningModule):
    def __init__(self, model, patch_size=256, overlap=32, batch_size=16):
        super().__init__()
        self.model = model
        self.criterion = nn.CrossEntropyLoss()
        self.f1 = MulticlassF1Score(
            num_classes=N_CLASSES,
            average="macro",
            ignore_index=None,
            multidim_average="global",
        )
        self.miou = MeanIoU(
            num_classes=N_CLASSES,
            include_background=True,
            per_class=False,
            input_format="index",
        )
        self.save_hyperparameters(ignore=["model"])

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        X, y = batch["image"], batch["mask"]
        logits = self(X)
        loss = self.criterion(logits, y)
        self.log(
            "train_loss", loss, on_step=True, on_epoch=True, prog_bar=True, logger=True
        )
        return loss

    def validation_step(self, batch, batch_idx):
        image, mask = (
            batch["image"].unsqueeze(-1).squeeze(0),
            batch["mask"].unsqueeze(-1),
        )

        subject = tio.Subject(
            image=tio.ScalarImage(tensor=image), mask=tio.LabelMap(tensor=mask)
        )
        sampler = tio.GridSampler(
            subject,
            patch_size=(self.hparams.patch_size, self.hparams.patch_size, 1),
            patch_overlap=(self.hparams.overlap, self.hparams.overlap, 0),
        )
        aggregator = tio.GridAggregator(sampler)
        with torch.no_grad():
            n_patches = 0
            total_loss = 0
            for patch in DataLoader(
                sampler, batch_size=self.hparams.batch_size, shuffle=False
            ):
                X, y = (
                    patch["image"][tio.DATA].to(self.device),
                    patch["mask"][tio.DATA].to(self.device),
                )
                X = X.squeeze(-1)
                y = y.squeeze(-1).squeeze(1).long()

                pred = self(X)
                aggregator.add_batch(
                    pred.detach().cpu().unsqueeze(-1), patch[tio.LOCATION]
                )

                loss = self.criterion(pred, y)
                total_loss += loss.item()
                n_patches += 1

        avg_loss = total_loss / n_patches
        self.log(
            "val_loss",
            avg_loss,
            on_epoch=True,
            prog_bar=True,
            batch_size=1,
            logger=True,
        )
        full_pred = aggregator.get_output_tensor().squeeze(-1)

        y_hat = full_pred.argmax(dim=0).unsqueeze(0).to(self.device)
        gt = mask.squeeze(-1).long()

        self.f1.update(y_hat, gt)
        self.miou.update(y_hat, gt)
        return None

    def on_validation_epoch_end(self):
        f1 = self.f1.compute()
        miou = self.miou.compute()

        self.log(
            "val_iou", miou, on_epoch=True, prog_bar=True, batch_size=1, logger=True
        )
        self.log("val_f1", f1, on_epoch=True, prog_bar=True, batch_size=1, logger=True)
        self.f1.reset()
        self.miou.reset()

    def on_fit_start(self):
        self.f1 = self.f1.to(self.device)
        self.miou = self.miou.to(self.device)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=1e-3, weight_decay=1e-4)
        return optimizer


Seed set to 42


In [3]:
x = torch.randn(1, IN_CHANNELS, 256, 256).to("cuda")

In [4]:
# segmentation_model.training_step(
#     batch={"image": x, "mask": torch.randint(0, N_CLASSES, (1, 256, 256)).to("cuda")},
#     batch_idx=0,
# )

In [5]:
# segmentation_model.validation_step(
#     batch={"image": x, "mask": torch.randint(0, N_CLASSES, (1, 256, 256)).to("cuda")},
#     batch_idx=0,
# )

In [6]:
OUTPUT_DIR = "Vaihingen_HF"
train_path = f"../{OUTPUT_DIR}/Vaihingen_train_patches-256x256/*"

val_path = f"../{OUTPUT_DIR}/Vaihingen_validation/*"

In [ ]:
from datasets import load_from_disk, concatenate_datasets
import glob


class HFDataModule(L.LightningDataModule):
    def __init__(self, train_path, val_path, batch_size=32):
        super().__init__()
        self.save_hyperparameters()

    def setup(self, stage=None):
        if stage == "fit":
            self.train_dataset = self._load_shards_into_dataset(self.hparams.train_path)

        if stage in ["fit", "validate"]:
            self.val_dataset = self._load_shards_into_dataset(self.hparams.val_path)

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.hparams.batch_size,
            shuffle=True,
            num_workers=10,
            pin_memory=True,
            prefetch_factor=1,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=1,
            shuffle=False,
            num_workers=10,
            pin_memory=True,
        )

    def _load_shards_into_dataset(self, path):
        paths = [load_from_disk(p) for p in glob.glob(path)]

        dataset = concatenate_datasets(paths)
        dataset.set_format(type="torch")
        return dataset


In [ ]:
from pathlib import Path

from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger

MODEL_NAME = "segnet_v1"
DATASET_NAME = "Vaihingen"
CHECKPOINT_PATH = None

checkpoint_callback = ModelCheckpoint(
    dirpath=f"l_checkpoints/{DATASET_NAME}/{MODEL_NAME}/",
    filename="{epoch:02d}-{step}-{val_iou:.3f}-{val_f1:.3f}-{val_loss:.3f}",
    save_top_k=1,
    monitor="val_iou",
    mode="max",  # IoU → maximize
    save_last=True,  # VERY IMPORTANT for resume
)

logger = TensorBoardLogger(
    save_dir="tb_logs", version=MODEL_NAME, name=DATASET_NAME, default_hp_metric=False
)


if CHECKPOINT_PATH is not None and not Path(CHECKPOINT_PATH).exists():
    raise RuntimeError("Checkpoint path does not exist")

segmentation_model = Segmentator(model, batch_size=8)
dm = DataModule(
    train_path=train_path,
    val_path=val_path,
    batch_size=segmentation_model.hparams.batch_size,
)

trainer = L.Trainer(
    max_epochs=10,
    accelerator="gpu",
    devices=1,
    enable_progress_bar=True,
    callbacks=[checkpoint_callback],
    accumulate_grad_batches=32,
    log_every_n_steps=1,
    precision="16-mixed",
    logger=logger,
)

trainer.fit(segmentation_model, datamodule=dm, ckpt_path=CHECKPOINT_PATH)

logger.log_hyperparams(
    segmentation_model.hparams,
    {
        "val_miou": trainer.callback_metrics["val_miou"].item(),
        "val_f1": trainer.callback_metrics["val_f1"].item(),
    },
)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/datacuber/Documents/semantic_segmentation/.venv/lib/python3.13/site-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ SegNet            │ 29.4 M │ train │     0 │
│ 1 │ criterion │ CrossEntropyLoss  │      0 │ train │     0 │
│ 2 │ f1        │ MulticlassF1Score │      0 │ train │     0 │
│ 3 │ miou      │ MeanIoU           │      0 │ train │     0 │
└───┴───────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 29.4 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 29.4 M                                                                                               
Total estimated model params size (MB): 117                                                                        
Modules in train mode: 57                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/home/datacuber/Documents/semantic_segmentation/.venv/lib/python3.13/site-packages/lightning/pytorch/utilities/_pyt
ree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and 
treespec.is_leaf()` instead.

`Trainer.fit` stopped: `max_epochs=10` reached.


AttributeError: 'SegNet' object has no attribute 'hparams'